# Sentiment Scoring — Baseline 2: FinBERT

Sentence-level financial sentiment scoring via ProsusAI/FinBERT, aggregated to a document-level score: mean(P(positive)) − mean(P(negative)) across sentences (spaCy sentence segmentation).

Note: FinBERT's positive/negative axis reflects general financial-news sentiment (e.g. growth data, macro conditions), not the hawkish/dovish policy-stance axis specifically — this is validated empirically in Step 4 rather than assumed.

## Step 0 — Path setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

## Step 1 — Dependency check

Requires `transformers` and `torch` (`pip install transformers torch`).

In [ ]:
import transformers
import torch
print("transformers version:", transformers.__version__)
print("torch version:", torch.__version__)

## Step 2 — Load the cleaned master dataset

In [ ]:
from paths import MASTER_TEXT_CSV

master = pd.read_csv(MASTER_TEXT_CSV, parse_dates=["listed_date"])
print(f"Loaded {len(master)} documents")

## Step 3 — Load models

spaCy (`en_core_web_sm`) for sentence segmentation; FinBERT (tokenizer + model) for sentiment scoring. FinBERT downloads (~440MB) and caches locally on first use.

In [ ]:
from sentiment import load_spacy_model, load_finbert

print("Loading spaCy...")
nlp = load_spacy_model()
print("Loading FinBERT (downloads on first run, please wait)...")
tokenizer, model = load_finbert()
print("Both models loaded.")

## Step 4 — Validation on known documents

Scores the same two reference documents used in Baseline 1 (Section 2.3 of the project report). Result: both score positive under FinBERT (Sep 2022: +0.269; Feb 2021: +0.393) — the dovish document scores *more* positive than the hawkish one, consistent with FinBERT tracking general economic-news sentiment rather than hawkish/dovish stance (see sentence-level inspection below).

In [ ]:
from sentiment import finbert_score_document, split_sentences, score_sentences_batch

sep2022 = master[(master["listed_date"] == "2022-09-30") & (master["doc_type"] == "resolution")].iloc[0]
feb2021 = master[(master["listed_date"] == "2021-02-05") & (master["doc_type"] == "resolution")].iloc[0]

print("Scoring Sep 2022 resolution (known hawkish)...")
score_sep2022 = finbert_score_document(sep2022["text"], nlp, tokenizer, model)
print(score_sep2022)

print("\nScoring Feb 2021 resolution (known dovish)...")
score_feb2021 = finbert_score_document(feb2021["text"], nlp, tokenizer, model)
print(score_feb2021)

Sentence-level inspection — which sentences drove the Sep 2022 document's score:

In [ ]:
sentences = split_sentences(sep2022["text"], nlp)
results = score_sentences_batch(sentences, tokenizer, model)
paired = sorted(zip(sentences, [r["positive"] - r["negative"] for r in results]), key=lambda x: x[1])

print("=== Most NEGATIVE sentences (Sep 2022) ===")
for s, sc in paired[:3]:
    print(f"[{sc:+.3f}] {s}")

print("\n=== Most POSITIVE sentences (Sep 2022) ===")
for s, sc in paired[-3:][::-1]:
    print(f"[{sc:+.3f}] {s}")

## Step 5 — Score full corpus

Sentence-level inference over ~24,000 sentences across 163 documents. CPU inference time is on the order of an hour; progress is logged every 10 documents.

In [ ]:
import time

finbert_rows = []
start = time.time()

for i, row in master.iterrows():
    result = finbert_score_document(row["text"], nlp, tokenizer, model)
    finbert_rows.append({"prid": row["prid"], **result})
    if (i + 1) % 10 == 0 or (i + 1) == len(master):
        elapsed = time.time() - start
        print(f"  {i + 1}/{len(master)} documents done ({elapsed:.0f}s elapsed)")

finbert_df = pd.DataFrame(finbert_rows)
print("\nDone.")
finbert_df.head()

## Step 6 — Merge with Baseline 1 and compare

In [ ]:
from paths import SENTIMENT_CSV, FIGURES_DIR

lexicon_scores = pd.read_csv(SENTIMENT_CSV)
combined = lexicon_scores.merge(finbert_df, on="prid", suffixes=("", "_finbert"))
combined.to_csv(SENTIMENT_CSV, index=False)
print(f"Saved combined scores -> {SENTIMENT_CSV}")
combined[["listed_date", "doc_type", "lexicon_score_x1000", "finbert_score"]].head(10)

In [ ]:
import matplotlib.pyplot as plt

res = combined[combined["doc_type"] == "resolution"].sort_values("listed_date")

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(res["listed_date"], res["lexicon_score_x1000"], color="tab:blue", marker="o", markersize=3, label="Lexicon (x1000)")
ax1.set_ylabel("Lexicon score (x1000 words)", color="tab:blue")
ax1.axhline(0, color="gray", linestyle="--", linewidth=1)

ax2 = ax1.twinx()
ax2.plot(res["listed_date"], res["finbert_score"], color="tab:red", marker="s", markersize=3, label="FinBERT")
ax2.set_ylabel("FinBERT score (pos - neg)", color="tab:red")

plt.title("RBI Resolution Sentiment: Lexicon vs FinBERT, 2016-2026")
fig.tight_layout()
plt.savefig(FIGURES_DIR / "lexicon_vs_finbert_timeseries.png", dpi=150)
plt.show()

---

Correlation between the lexicon and FinBERT scores across all 61 Resolutions: 0.29 — positive but modest, consistent with the two measures capturing related but distinct signals (policy-stance tone vs. general economic sentiment).